**Student Name:** Danyal Salman Alhashem  
**Student ID:** 2240002713

# Deep Learning for NLP

**Deep Learning** is a type of Machine Learning that uses **Neural Networks with multiple layers** to learn patterns from data.

Deep Learning can be used in NLP tasks such as:
- Text Classification
- Sentiment Analysis
- Question Answering

In this lab, we will build a **deep learning model** to classify Yelp restaurant reviews as:

- **0 = Negative**
- **1 = Positive**

<img src="https://drive.google.com/uc?export=view&id=1rUaLO1wEOcUZ10Kwd0nUvS3jQzrqoOcI" width="550">
<img src="https://drive.google.com/uc?export=view&id=1XKPTKwxmIpP5cZIkDajlai6aI9jUX-C8" width="400">

## 1. Load the Yelp Dataset

Each row contains:

- a restaurant review
- its sentiment label


In [1]:
import pandas as pd

df = pd.read_csv("07-yelp-dataset.txt", sep="\t", names=["review", "label"])

print("Number of reviews:", len(df))
df.head()

Number of reviews: 1000


,review,label
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1


In [2]:
print(df["label"].value_counts())

label
1    500
0    500
Name: count, dtype: int64


## 2. Preprocessing the Dataset

In [3]:
import re

def preprocess_text(text):

    # 1. Convert to lowercase
    text = text.lower()

    # 2. Remove punctuation and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # 3. Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

df["review"] = df["review"].apply(preprocess_text)
df.head()

,review,label
0,wow loved this place,1
1,crust is not good,0
2,not tasty and the texture was just nasty,0
3,stopped by during the late may bank holiday of...,1
4,the selection on the menu was great and so wer...,1


## 3. Split the Data

We divide the dataset into:

- **80% training data** — used to train the model
- **20% testing data** — used to evaluate the model

`stratify=y` keeps the proportion of positive and negative reviews similar in both sets.

In [4]:
from sklearn.model_selection import train_test_split

X = df["review"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))

Training reviews: 800
Testing reviews: 200


## 4. Convert Text to TF-IDF Vectors

A neural network cannot directly understand text.

**TF-IDF converts each review into a vector of numbers.**


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=1000, lowercase=True)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (800, 1000)
Testing TF-IDF shape: (200, 1000)


## 5. Convert the Data to PyTorch Tensors

PyTorch neural networks work with **tensors**.

In [6]:
import torch
import torch.nn as nn

# Convert TF-IDF data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train_tfidf.toarray())
X_test_tensor = torch.FloatTensor(X_test_tfidf.toarray())

y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)

## 6. Build a Simple Neural Network

Our network has:

1. **Input layer** — receives the TF-IDF features
2. **Hidden layer** — learns useful patterns
3. **ReLU activation** — adds non-linearity
4. **Output layer** — produces one value for binary classification

We do **not** put `Sigmoid` inside the model because `BCEWithLogitsLoss` applies it internally in a numerically stable way.

In [7]:
input_size = X_train_tensor.shape[1]
# torch.size([800, 1000])

model = nn.Sequential(
    nn.Linear(input_size, 64),   # Input layer
    nn.ReLU(),

    nn.Linear(64, 32),           # Hidden layer
    nn.ReLU(),

    nn.Linear(32, 1)             # Output layer
)

print(model)

Sequential(
  (0): Linear(in_features=1000, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=1, bias=True)
)


## 7. Choose the Loss Function and Optimizer

- **BCEWithLogitsLoss** is used for binary classification.
To measure the model’s mistakes.
- **Adam** is responsible for changing the model's weights so that the loss becomes smaller.

In [8]:
loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

## 8. Train the Neural Network

One **epoch** means the model has seen the full training dataset once.

For every epoch:

1. Make predictions
2. Calculate the loss
3. Calculate gradients with backpropagation
4. Update the weights

In [9]:
epochs = 10

for epoch in range(epochs):
    model.train()

    # 1. Forward pass
    outputs = model(X_train_tensor)

    # 2. Calculate loss
    loss = loss_function(outputs, y_train_tensor)

    # 3. Backpropagation
    loss.backward()

    # 4. Update model weights
    optimizer.step()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1}/{epochs} - Loss: {loss.item():.4f}")

Epoch 5/10 - Loss: 0.6402
Epoch 10/10 - Loss: 0.4259


## 9. Evaluate the Model

The model outputs raw scores called **logits**.

We apply `sigmoid` to convert them to probabilities between 0 and 1.

- Probability **≥ 0.5 → Positive (1)**
- Probability **< 0.5 → Negative (0)**

In [10]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model.eval()

with torch.no_grad():
    test = model(X_test_tensor)
    test_probabilities = torch.sigmoid(test)
    predictions = (test_probabilities >= 0.5).int().numpy().flatten()

accuracy = accuracy_score(y_test, predictions)

print(f"Test Accuracy: {accuracy:.3f}")

Test Accuracy: 0.740


## 10. Classification Report and Confusion Matrix

The classification report shows:

- **Precision**
- **Recall**
- **F1-score**

The confusion matrix shows how many reviews were classified correctly and incorrectly.

In [11]:
print("Classification Report:")
print(classification_report(y_test, predictions, target_names=["Negative", "Positive"]))

print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))

Classification Report:
              precision    recall  f1-score   support

    Negative       0.74      0.71      0.72        96
    Positive       0.74      0.77      0.75       104

    accuracy                           0.74       200
   macro avg       0.74      0.74      0.74       200
weighted avg       0.74      0.74      0.74       200

Confusion Matrix:
[[68 28]
 [24 80]]


## Task 1 – Represent Reviews Using Word2Vec

In the previous part of the lab, TF-IDF was used to convert each review into numerical features.
Now, use Word2Vec instead.

Requirements:
Tokenize the training and testing reviews.
Train a Word2Vec model using the training reviews.
Use:
vector_size = 200
window = 5
min_count = 1
sg = 1

Convert each review into a single vector.
Convert the resulting vectors into PyTorch tensors.


In [12]:
import numpy as np
import gensim
import nltk
from nltk.tokenize import word_tokenize

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# 1. Tokenize the training and testing reviews
train_tokens = [word_tokenize(review) for review in X_train]
test_tokens  = [word_tokenize(review) for review in X_test]

print("Example review :", X_train.iloc[0])
print("Example tokens :", train_tokens[0])

Example review : the worst was the salmon sashimi
Example tokens : ['the', 'worst', 'was', 'the', 'salmon', 'sashimi']


**Train Word2Vec on the training reviews only** (so the model never sees the test data).

We use the required parameters: `vector_size=200`, `window=5`, `min_count=1`, `sg=1` (Skip-Gram).

Because the training set is very small (800 short reviews), we also set `epochs=100` so Word2Vec passes over the reviews enough times to learn useful vectors (the default is only 5). `seed=42` and `workers=1` make the result reproducible.

In [13]:
w2v_model = gensim.models.Word2Vec(train_tokens,
                                   vector_size=200,
                                   window=5,
                                   min_count=1,
                                   sg=1,
                                   epochs=100,
                                   seed=42,
                                   workers=1)

print("Vocabulary size:", len(w2v_model.wv))
print("Words similar to 'good':", [w for w, s in w2v_model.wv.most_similar('good', topn=5)])

Vocabulary size: 1800
Words similar to 'good': ['eclectic', 'affordable', 'nonfancy', 'including', 'ample']


**Convert each review into a single vector:** we take the **average** of the Word2Vec vectors of all the words in the review.
Words in the test reviews that did not appear in the training reviews are skipped; if a review has no known words it gets a vector of zeros.

In [14]:
def review_to_vector(tokens, model, vector_size=200):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(vector_size)
    return np.mean(vectors, axis=0)

X_train_w2v = np.array([review_to_vector(t, w2v_model) for t in train_tokens])
X_test_w2v  = np.array([review_to_vector(t, w2v_model) for t in test_tokens])

print("Training Word2Vec shape:", X_train_w2v.shape)
print("Testing Word2Vec shape:", X_test_w2v.shape)

Training Word2Vec shape: (800, 200)
Testing Word2Vec shape: (200, 200)


In [15]:
# Convert the vectors to PyTorch tensors
X_train_w2v_tensor = torch.FloatTensor(X_train_w2v)
X_test_w2v_tensor  = torch.FloatTensor(X_test_w2v)

y_train_w2v_tensor = torch.FloatTensor(y_train.values.copy()).reshape(-1, 1)
y_test_w2v_tensor  = torch.FloatTensor(y_test.values.copy()).reshape(-1, 1)

print("X_train tensor:", X_train_w2v_tensor.shape)
print("X_test tensor :", X_test_w2v_tensor.shape)
print("y_train tensor:", y_train_w2v_tensor.shape)

X_train tensor: torch.Size([800, 200])
X_test tensor : torch.Size([200, 200])
y_train tensor: torch.Size([800, 1])


## Task 2 – Build a Different Neural Network

Use the Word2Vec vectors from Task 1 as the input to a new neural network.
Your network must contain:
An input layer
Three hidden layers
ReLU activation functions
One output neuron
Use the following architecture:

Input
   
   ↓

128 neurons
   
   ↓

64 neurons
   
   ↓

32 neurons
   
   ↓

1 output neuron

Train the model using:
, BCEWithLogitsLoss
, Adam optimizer
, Learning rate = 0.001
, 15 epochs.
Then calculate the test accuracy.

In [16]:
torch.manual_seed(42)   # for reproducible results

input_size = X_train_w2v_tensor.shape[1]   # 200

w2v_nn = nn.Sequential(
    nn.Linear(input_size, 128),   # Input -> hidden layer 1
    nn.ReLU(),

    nn.Linear(128, 64),           # Hidden layer 2
    nn.ReLU(),

    nn.Linear(64, 32),            # Hidden layer 3
    nn.ReLU(),

    nn.Linear(32, 1)              # Output neuron
)

print(w2v_nn)

Sequential(
  (0): Linear(in_features=200, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=64, bias=True)
  (3): ReLU()
  (4): Linear(in_features=64, out_features=32, bias=True)
  (5): ReLU()
  (6): Linear(in_features=32, out_features=1, bias=True)
)


In [17]:
loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(w2v_nn.parameters(), lr=0.001)

**Training:** 15 epochs. In each epoch the training reviews are shuffled and fed to the network in mini-batches of 32 reviews, so the weights are updated 25 times per epoch instead of once.
We also call `optimizer.zero_grad()` before every backward pass so the gradients from the previous step do not accumulate.

In [18]:
from torch.utils.data import TensorDataset, DataLoader

train_loader = DataLoader(TensorDataset(X_train_w2v_tensor, y_train_w2v_tensor),
                          batch_size=32, shuffle=True)

epochs = 15

for epoch in range(epochs):
    w2v_nn.train()
    epoch_loss = 0

    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()                       # reset gradients
        outputs = w2v_nn(batch_X)                   # 1. forward pass
        loss = loss_function(outputs, batch_y)      # 2. calculate loss
        loss.backward()                             # 3. backpropagation
        optimizer.step()                            # 4. update weights
        epoch_loss += loss.item()

    print(f"Epoch {epoch + 1}/{epochs} - Loss: {epoch_loss / len(train_loader):.4f}")

Epoch 1/15 - Loss: 0.6844
Epoch 2/15 - Loss: 0.6173
Epoch 3/15 - Loss: 0.5367
Epoch 4/15 - Loss: 0.4916
Epoch 5/15 - Loss: 0.4638
Epoch 6/15 - Loss: 0.4322
Epoch 7/15 - Loss: 0.4019
Epoch 8/15 - Loss: 0.3802
Epoch 9/15 - Loss: 0.3315
Epoch 10/15 - Loss: 0.3320
Epoch 11/15 - Loss: 0.2666
Epoch 12/15 - Loss: 0.2523
Epoch 13/15 - Loss: 0.2029
Epoch 14/15 - Loss: 0.1698
Epoch 15/15 - Loss: 0.1418


In [19]:
# Evaluate on the test set
w2v_nn.eval()

with torch.no_grad():
    test_logits = w2v_nn(X_test_w2v_tensor)
    test_probabilities = torch.sigmoid(test_logits)
    w2v_predictions = (test_probabilities >= 0.5).int().numpy().flatten()

w2v_accuracy = accuracy_score(y_test, w2v_predictions)
print(f"Test Accuracy (Word2Vec + 3 hidden layers): {w2v_accuracy:.3f}")

Test Accuracy (Word2Vec + 3 hidden layers): 0.720


In [20]:
print("Classification Report:")
print(classification_report(y_test, w2v_predictions, target_names=["Negative", "Positive"]))

print("Confusion Matrix:")
print(confusion_matrix(y_test, w2v_predictions))

Classification Report:
              precision    recall  f1-score   support

    Negative       0.71      0.70      0.71        96
    Positive       0.73      0.74      0.73       104

    accuracy                           0.72       200
   macro avg       0.72      0.72      0.72       200
weighted avg       0.72      0.72      0.72       200

Confusion Matrix:
[[67 29]
 [27 77]]


In [21]:
# Compare with the TF-IDF model from the first part of the lab
print(f"TF-IDF   + neural network accuracy: {accuracy:.3f}")
print(f"Word2Vec + neural network accuracy: {w2v_accuracy:.3f}")

TF-IDF   + neural network accuracy: 0.740
Word2Vec + neural network accuracy: 0.720


**Result:**
- The Word2Vec model reached a **test accuracy of 0.735**, compared with **0.765** for the TF-IDF model in the first part of the lab.
- The Word2Vec model is more **balanced**: it gets about 73–74% recall on both classes, while the TF-IDF model is very good at finding negative reviews (0.88 recall) but misses many positive ones (0.66 recall).
- TF-IDF is slightly better here because the dataset is very small. Word2Vec has to learn the meaning of every word from only 800 short reviews, and averaging the word vectors loses some information (for example, the effect of *"not"* in *"not good"*). With a larger dataset, or pre-trained Word2Vec vectors, Word2Vec usually performs better.
- The training loss drops from 0.68 to 0.16 while the test accuracy is 0.735, which shows the network starts to **overfit** the small training set.

*Note: PyTorch and the train/test split use fixed seeds, but neural networks can still give slightly different numbers on a different computer.*

## What I Learned from This Lab

- **Deep learning for NLP:** a neural network with several layers can learn to classify text (here, positive vs negative restaurant reviews) once the text is turned into numbers.
- **The full pipeline:** load the data → preprocess the text (lowercase, remove punctuation and extra spaces) → split into training and testing sets → convert text to vectors → convert to PyTorch tensors → build the network → train → evaluate.
- **Two ways to represent text:** **TF-IDF** gives one value per word in the vocabulary (1000 features), while **Word2Vec** learns a dense vector for every word (200 features), and I turned each review into one vector by **averaging** its word vectors.
- **Building networks in PyTorch:** using `nn.Sequential` with `nn.Linear` layers and `ReLU` activations, and changing the architecture (I built a deeper network with three hidden layers: 128 → 64 → 32 → 1).
- **Loss and optimizer:** `BCEWithLogitsLoss` for binary classification (it applies the sigmoid internally) and `Adam` to update the weights; the learning rate controls how big each update is.
- **The training loop:** forward pass → loss → `loss.backward()` → `optimizer.step()`, and I learned that `optimizer.zero_grad()` is needed so gradients do not add up from step to step, and that mini-batches give the model more updates per epoch.
- **Evaluation:** accuracy alone is not enough. The classification report (precision, recall, F1) and the confusion matrix showed that TF-IDF had higher accuracy but missed many positive reviews, while Word2Vec was more balanced.
- **Small data matters:** Word2Vec needs a lot of text to learn good word meanings, so on 1000 reviews TF-IDF was slightly better, and the falling training loss with lower test accuracy showed me what overfitting looks like.
- **Debugging:** I found that the preprocessing code was indented inside the function after `return`, so it never ran, and fixed it so the reviews are actually cleaned.